# Sending Your First Requests

In this notebook, you will learn how to interact with an llm-d deployment using
the **OpenAI-compatible API**. llm-d exposes the same `/v1/chat/completions`
endpoint that you already know from OpenAI, so any existing OpenAI SDK code
works with minimal changes.

We will cover:
- Setting up the Python client
- Sending a basic chat completion
- Streaming responses token-by-token
- Multi-turn conversations (and why prefix caching makes them fast)
- Checking metrics to observe cache hit rates

In [ ]:
# Install the OpenAI Python SDK
!pip install openai

In [ ]:
from openai import OpenAI

# Point the OpenAI client at your llm-d gateway.
# Replace the base_url with your actual gateway endpoint.
# The api_key can be any string since llm-d does not require auth by default.

GATEWAY_URL = "http://localhost:8080"  # or your cluster's external IP
MODEL_NAME = "Qwen/Qwen3-32B"

client = OpenAI(
    base_url=f"{GATEWAY_URL}/v1",
    api_key="not-needed",  # llm-d doesn't require an API key by default
)

print(f"Client configured to connect to: {GATEWAY_URL}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# Send a simple chat completion request
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what llm-d is in three sentences."},
    ],
    max_tokens=200,
    temperature=0.7,
)

# Print the response
print("=== Response ===")
print(response.choices[0].message.content)
print()
print("=== Usage ===")
print(f"  Prompt tokens:     {response.usage.prompt_tokens}")
print(f"  Completion tokens: {response.usage.completion_tokens}")
print(f"  Total tokens:      {response.usage.total_tokens}")

## Understanding the Response

The response object follows the OpenAI Chat Completions format:

- **`choices`** - A list of generated completions. Each choice has:
  - `message.content` - The generated text
  - `finish_reason` - Why generation stopped (`stop`, `length`, etc.)
- **`usage`** - Token counts for billing and monitoring:
  - `prompt_tokens` - Tokens in your input messages
  - `completion_tokens` - Tokens generated by the model
- **`model`** - The model that served the request
- **`id`** - A unique request ID for debugging

Because llm-d is API-compatible with OpenAI, you can swap in llm-d as a drop-in
replacement for any application that uses the OpenAI SDK.

In [ ]:
# Send a streaming request - tokens arrive as they are generated
import sys

print("=== Streaming Response ===")

stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Write a haiku about distributed inference."},
    ],
    max_tokens=50,
    stream=True,
)

full_response = ""
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        full_response += token
        sys.stdout.write(token)
        sys.stdout.flush()

print("\n")
print(f"Total tokens received: {len(full_response.split())} words")

## Streaming vs Non-Streaming

| | Non-Streaming | Streaming |
|---|---|---|
| **Response** | Full text returned at once | Tokens arrive incrementally |
| **Time to first byte** | Must wait for full generation | First token arrives quickly |
| **Use case** | Batch processing, APIs | Chat UIs, real-time apps |
| **Parameter** | `stream=False` (default) | `stream=True` |

Streaming is essential for interactive applications. The user sees tokens appear
as they are generated, reducing perceived latency. Under the hood, llm-d uses
Server-Sent Events (SSE) to push each token chunk to the client.

In [ ]:
# Multi-turn conversation - demonstrates prefix caching benefits
# The shared prefix (system message + earlier turns) gets cached on the model server,
# so follow-up turns have much lower TTFT.

import time

conversation = [
    {"role": "system", "content": "You are an expert on Kubernetes and cloud-native infrastructure. Give concise answers."},
]

questions = [
    "What is a Pod in Kubernetes?",
    "How does a Deployment differ from a StatefulSet?",
    "What are GPU resource requests and limits?",
]

for i, question in enumerate(questions):
    conversation.append({"role": "user", "content": question})

    start = time.time()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=conversation,
        max_tokens=150,
    )
    elapsed = time.time() - start

    answer = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": answer})

    print(f"--- Turn {i + 1} ({elapsed:.2f}s) ---")
    print(f"Q: {question}")
    print(f"A: {answer}")
    print()

print("Notice: Turn 2 and 3 should be faster than Turn 1 because the shared")
print("prefix (system prompt + prior turns) is already cached on the model server.")
print("The EPP router directs follow-up requests to the same replica that holds the cache.")

In [ ]:
# Check metrics endpoint to observe prefix cache hit rate
# The EPP exposes Prometheus metrics that reveal routing decisions

import urllib.request
import json

EPP_METRICS_URL = f"{GATEWAY_URL}/metrics"  # Adjust if EPP metrics are on a different port

try:
    with urllib.request.urlopen(EPP_METRICS_URL, timeout=5) as resp:
        metrics_text = resp.read().decode("utf-8")

    # Filter for cache-related metrics
    print("=== Prefix Cache Metrics ===")
    for line in metrics_text.split("\n"):
        if "prefix_cache" in line.lower() or "cache_hit" in line.lower():
            print(line)

    # Filter for routing metrics
    print("\n=== Routing Metrics ===")
    for line in metrics_text.split("\n"):
        if "epp_request" in line.lower() or "endpoint_picked" in line.lower():
            print(line)

except Exception as e:
    print(f"Could not reach metrics endpoint: {e}")
    print("Tip: You may need to port-forward the EPP metrics port:")
    print("  kubectl port-forward svc/epp -n llm-d 9090:9090")

## Summary

You have learned how to:
1. Connect to llm-d using the standard OpenAI Python SDK
2. Send chat completion requests (both streaming and non-streaming)
3. Build multi-turn conversations that benefit from prefix caching
4. Check metrics to verify that caching and routing are working

### Next Steps

- **Optimized Baseline** - Dive deeper into how prefix-cache routing and
  load-aware scheduling work, including benchmarking.
- **P/D Disaggregation** - Learn how to separate prefill and decode for
  maximum throughput on large models.
- **Flow Control** - Add priority bands and fairness policies for
  multi-tenant production workloads.